## Task 3: Monthly Install Trend Analysis
**Author:** Vansh Sharma


- Cleaned and prepared the Play Store dataset.
- Applied app name, category, and review-based filters.
- Calculated monthly installs and month-over-month growth.
- Highlighted periods with more than 20% install growth.
- Translated selected category names for graph display.
- Displayed the graph only between **6 PM IST and 9 PM IST**.


#  Import Libraries
 

In [26]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime
import matplotlib.pyplot as plt
from zoneinfo import ZoneInfo
from datetime import datetime
import pytz

# Load Dataset `play_store_data`

In [27]:
play_store_data = pd.read_csv(
    r"C:\Users\Vansh Sharma\Downloads\Play Store Data (1).csv"
)

pd.DataFrame(play_store_data.head(3))

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up


### Data Cleaning

- Removed corrupted records.
- Converted Installs and Reviews into numeric format.
- Converted Last Updated into datetime format.
- Handled missing values using forward fill and mode imputation.

In [29]:
# Remove corrupted row
play_store_data = play_store_data[
    play_store_data["Installs"] != "Free"
]


# Clean Installs column
play_store_data["Installs"] = (
    play_store_data["Installs"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
    .astype(int)
)


# Convert Reviews into integer
play_store_data["Reviews"] = (
    play_store_data["Reviews"]
    .astype(int)
)


# Convert Last Updated into datetime
play_store_data["Last Updated"] = pd.to_datetime(
    play_store_data["Last Updated"],
    errors="coerce"
)


# Fill missing values

play_store_data["Rating"] = (
    play_store_data["Rating"]
    .ffill()
)


play_store_data["Current Ver"] = (
    play_store_data["Current Ver"]
    .ffill()
)


play_store_data["Android Ver"] = (
    play_store_data["Android Ver"]
    .ffill()
)


play_store_data["Type"] = (
    play_store_data["Type"]
    .fillna(play_store_data["Type"].mode()[0])
)


 

### Data Filtering

- Removed apps starting with X, Y, or Z.
- Selected categories beginning with E, C, or B.
- Kept apps with more than 500 reviews.
- Excluded app names containing the letter "S".

In [30]:
# Category starts with E, C, B

play_store_data = play_store_data[
    play_store_data["Category"].str.startswith(
        ("E", "C", "B")
    )
]


# App name should not start with X, Y, Z

play_store_data = play_store_data[
    ~play_store_data["App"].str.startswith(
        ("X", "Y", "Z"),
        na=False
    )
]


# Reviews greater than 500

play_store_data = play_store_data[
    play_store_data["Reviews"] > 500
]


# App name should not contain letter S

play_store_data = play_store_data[
    ~play_store_data["App"].str.contains(
        "S",
        case=False,
        na=False
    )
]


# Check result

play_store_data.shape

(245, 13)

### Monthly Install Trend

- Extracted month from the Last Updated column.
- Calculated total installs for each category on a monthly basis.
- Computed month-over-month growth percentage.

In [31]:
# Extract month from Last Updated

play_store_data["Month"] = (
    play_store_data["Last Updated"]
    .dt.to_period("M")
    .astype(str)
)

In [32]:
monthly_installs = (
    play_store_data
    .groupby(
        ["Month", "Category"],
        as_index=False
    )
    ["Installs"]
    .sum()
)

monthly_installs.head()

,Month,Category,Installs
0,2014-01,COMMUNICATION,100000
1,2014-03,COMMUNICATION,100000
2,2014-05,BUSINESS,100000
3,2014-07,COMMUNICATION,10000000
4,2014-07,EDUCATION,10000


In [33]:
# Sort data

monthly_installs = monthly_installs.sort_values(
    ["Category", "Month"]
)


# Calculate month-over-month growth %

monthly_installs["MoM_Growth"] = (
    monthly_installs
    .groupby("Category")["Installs"]
    .pct_change()
    * 100
)


# Round values

monthly_installs["MoM_Growth"] = (
    monthly_installs["MoM_Growth"]
    .round(2)
)


monthly_installs.head()

,Month,Category,Installs,MoM_Growth
46,2018-03,BEAUTY,5000,NaN
58,2018-06,BEAUTY,1000000,19900.0
64,2018-07,BEAUTY,1000000,0.0
72,2018-08,BEAUTY,100000,-90.0
5,2014-10,BOOKS_AND_REFERENCE,500000,NaN


### Category Translation

- Displayed Beauty in Hindi.
- Displayed Business in Tamil.
- Displayed Dating in German.

In [34]:
category_translation = {
    "BEAUTY": "सौंदर्य",
    "BUSINESS": "வணிகம்",
    "DATING": "Partnersuche"
}


monthly_installs["Category_Label"] = (
    monthly_installs["Category"]
    .replace(category_translation)
)


monthly_installs.head()

,Month,Category,Installs,MoM_Growth,Category_Label
46,2018-03,BEAUTY,5000,NaN,सौंदर्य
58,2018-06,BEAUTY,1000000,19900.0,सौंदर्य
64,2018-07,BEAUTY,1000000,0.0,सौंदर्य
72,2018-08,BEAUTY,100000,-90.0,सौंदर्य
5,2014-10,BOOKS_AND_REFERENCE,500000,NaN,BOOKS_AND_REFERENCE


### Time-Based Visualization

- Created a line chart showing monthly installs by category.
- Highlighted periods with more than 20% month-over-month growth.
- Restricted graph visibility to **6 PM IST – 9 PM IST**.

In [37]:
# Current IST Time
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

# Allowed Time
start_time = datetime.strptime("18:00", "%H:%M").time()
end_time = datetime.strptime("21:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    plt.figure(figsize=(14,6))

    for category in monthly_installs["Category_Label"].unique():

        data = (
            monthly_installs[
                monthly_installs["Category_Label"] == category
            ]
            .sort_values("Month")
        )

        # Line Plot
        plt.plot(
            data["Month"],
            data["Installs"],
            marker="o",
            linewidth=2,
            label=category
        )

        # Highlight Growth > 20%
        high_growth = data["MoM_Growth"] > 20

        plt.fill_between(
            data["Month"],
            data["Installs"],
            where=high_growth,
            alpha=0.3,
            interpolate=True
        )

    plt.title(
        "Monthly Installs Trend by App Category"
    )

    plt.xlabel("Month")
    plt.ylabel("Total Installs")

    plt.xticks(rotation=45)

    plt.legend(title="Category")

    plt.grid(axis="y")

    plt.tight_layout()

    plt.show()

else:
    print(
        "Graph is available only between 6 PM IST and 9 PM IST"
    )

Graph is available only between 6 PM IST and 9 PM IST


# `KPIs`

In [36]:
total_categories = monthly_installs["Category"].nunique()

top_category = (
    monthly_installs.groupby("Category")["Installs"]
    .sum()
    .idxmax()
)

total_installs = monthly_installs["Installs"].sum()

avg_growth = (
    monthly_installs["MoM_Growth"]
    .mean()
    .round(2)
)

high_growth_months = (
    monthly_installs["MoM_Growth"] > 20
).sum()

print("Total Categories:", total_categories)
print("Top Category:", top_category)
print("Total Installs:", total_installs)
print("Average MoM Growth (%):", avg_growth)
print("High Growth Periods (>20%):", high_growth_months)

Total Categories: 8
Top Category: COMMUNICATION
Total Installs: 5947156000
Average MoM Growth (%): 1516.89
High Growth Periods (>20%): 37


### Key Performance Indicators (KPIs)

- **Total Categories Analyzed:** 8

- **Top Performing Category:** Communication

- **Total Installs:** 5.95 Billion

- **Average Month-over-Month Growth:** 1516.89%

- **High Growth Periods (>20%):** 37